In [1]:
%pip install -q pypdf faiss-cpu ollama ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
from io import BytesIO
from pypdf import PdfReader
from ipywidgets import widgets
from IPython.display import display
import ollama
import faiss
import numpy as np

EMBED_MODEL = "qwen3-embedding:latest"
CHAT_MODEL = "qwen3:1.7b"

print("All library are installed successfully")

All library are installed successfully


In [2]:
def load_and_chunk_pdf(pdf_bytes, chunk_size=1000, overlap=150):
    reader = PdfReader(BytesIO(pdf_bytes))

    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    chunks = []
    start = 0

    while start < len(full_text):
        end = start + chunk_size
        chunk = full_text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [3]:
uploader = widgets.FileUpload(accept='.pdf', multiple=False, description="Upload PDF") 
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload PDF')

In [4]:
uploaded_files = uploader.value
if not uploaded_files:
    print("No file uploaded.")

if isinstance(uploaded_files, dict):
    uploaded_file =next(iter(uploaded_files.values())) 
    pdf_name= uploaded_file["metadata"] ["name"]
    

else:
    uploaded_file = uploaded_files [0]
    pdf_name = uploaded_file ["name"]

pdf_bytes = bytes(uploaded_file["content"])
print(f"Uploaded PDF: {pdf_name}")


pdf_chunks = load_and_chunk_pdf(pdf_bytes)
print("PDF loaded and chunked into", len (pdf_chunks), "chunks.")
print("First chunk preview:",
pdf_chunks [0][:200], "...") # Print first 200 characters of the first chunk

Uploaded PDF: AI_DS_Scheme_plus_Syllabus_all_years_2022_april.pdf
PDF loaded and chunked into 254 chunks.
First chunk preview: 1 
 
 
 
 
 
 
 
 
SCHEME & SYLLABUS 
 
 
OF 
 
 
B.TECH. (ARTIFICIAL INTELLIGENCE AND DATA SCIENCE) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2 
 
 
FIRST YEA ...


In [5]:
import time
EMBED_MODEL = "qwen3-embedding:latest"
CHAT_MODEL = "qwen3:1.7b"

print("Embedding Model:", EMBED_MODEL)
print("Chat Model:", CHAT_MODEL)


def embed_chunks(texts, batch_size=2, retries=3):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        for attempt in range(retries):
            try:
                response = ollama.embed(model=EMBED_MODEL,input=batch)
                all_embeddings.extend(response["embeddings"])
                break

            except Exception as e:
                print(f"Error embedding batch {i // batch_size + 1}: {e}")
                if attempt < retries - 1:
                    time.sleep(2)
                else:
                    raise

    return all_embeddings

chunk_embeddings = embed_chunks(pdf_chunks)
chunk_embeddings = np.array(chunk_embeddings, dtype="float32")

print("Embeddings generated:", chunk_embeddings.shape)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index created. Total vectors:", index.ntotal)

Embedding Model: qwen3-embedding:latest
Chat Model: qwen3:1.7b
Embeddings generated: (254, 4096)
FAISS index created. Total vectors: 254


In [6]:
def retrive_relevant_chunks(query, top_k=8):
    query_embedding = embed_chunks([query])

    distance, indices = index.search(np.array(query_embedding).astype("float32"),top_k)
    relevant_chunks = [pdf_chunks[i]for i in indices[0]]

    return relevant_chunks


text_question = "What is this document about?"

found_chunks = retrive_relevant_chunks(text_question)

print("Found relevant chunks for the question:", text_question)

for i, chunk in enumerate(found_chunks, 1):
    print(f"Chunk {i} preview:", chunk[:200], "...") # print first 200 character of each releveant chunk


Found relevant chunks for the question: What is this document about?
Chunk 1 preview: dents will be able to: 
CO1  Understand basic grammar principles, and apply them to synthesise and transform 
sentences and  identify common errors in writing 
CO2  Demonstrate enhanced communicative  ...
Chunk 2 preview: arty for Miss Pushpa T.S. – Poem 
(Introduction to Indianisms and Difference between Indian English and Standard English) 
 
George Orwell – Politics and the English Language – Essay 
(Writing process ...
Chunk 3 preview: A Tidy Approach”, Julia Silge, David Robinson, O’Reilly 
2. “Text Analysis with R for Students of Literature”, Matthew L. Jockers, Springer. 
3. “Natural Language Annotation for Machine Learning”, Jam ...
Chunk 4 preview: hanics  2 0 1 2 0 2 50 30 20 
7. ESC EE 100 (ESC)  Electrical Engineering 3 0 1 3 0 2 50 30 20 
8. HSMC REE100 
(HSM) 
Environmental Studies 
and Disaster Management 
2 0 0 2 0 0 80 0 20 
   Total 9 0 ...
Chunk 5 preview: are Quality Assurance: Qua

In [8]:
def ask_pdf(question, top_k=8):
    relevant_chunks = retrive_relevant_chunks(question, top_k)
    context = "\n\n---\n\n".join(relevant_chunks)

    prompt = f"""You are answering questions about a document using the excerpts below. Read all the excerpts carefully; the answer may be spread across multiple excerpts.

Excerpts:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={"num_ctx": 4096}
    )

    return response["message"]["content"]


answer = ask_pdf("What is this document about?")
print("Answer:", answer)

Answer: The document is a comprehensive course curriculum and syllabus for **PE-II(e): INTRODUCTION TO VIRTUAL & AUGMENTED REALITY** and related interdisciplinary subjects. It outlines the structure, learning outcomes, and practical components of the course, along with supplementary materials and references. Key themes include:

1. **Course Structure**:  
   - Focuses on **virtual and augmented reality (VR/AR)**, emphasizing **algorithmic foundations**, **application development**, and **technical requirements** for modern systems.  
   - Includes units on **software engineering**, **text mining**, **computer security**, **forensic investigations**, and **data analysis**.

2. **Learning Outcomes**:  
   - **CO1–CO3** (understand system requirements, algorithms, and application development for VR/AR).  
   - **Software Engineering** (use case models, UML diagrams, testing, and quality assurance).  
   - **Text Mining** (features, clustering, NLP, and data analysis).  
   - **Computer Se

In [9]:
question_box = widgets.Text(
    description="Enter your question:",
    placeholder="Type your question here...",
    layout=widgets.Layout(width="80%"),
    continuous_update=False
)

ask_button = widgets.Button(description="Ask PDF",button_style="primary")
chat_output = widgets.Output()


def handle_questions(_):
    question = question_box.value.strip()

    if not question:
        return
    question_box.value = ""

    with chat_output:
        print(f"Question: {question}")
        answer = ask_pdf(question)
        print(f"Answer: {answer}")

ask_button.on_click(handle_questions)
question_box.observe(handle_questions, names="value")

display(widgets.HBox([question_box, ask_button]),chat_output)

Output()